# 🌽 Safrinha 2025 — Retreinamento seguro com checkpoints

Este notebook replica o fluxo anterior de retreinamento e validação visual, mas isolando os artefatos da **safrinha de 2025** para evitar sobrescrever modelos/checkpoints da safra principal.

Caso o arquivo da safrinha não tenha `SAFRINHA`/`2025` no nome, preencha `CAMINHO_TFRECORD_SAFRINHA` com o caminho completo. Se esse campo ficar vazio, o notebook busca TFRecords em `pasta_base` e subpastas, sem diferenciar maiúsculas/minúsculas, e usa um fallback para o melhor arquivo encontrado.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

# 1. Montar Drive (Obrigatório no Colab)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURAÇÕES DA SAFRINHA 2025 ---
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
NOME_CENARIO = 'Safrinha_2025'

# Se a busca automática não encontrar o arquivo correto, cole o caminho completo aqui.
# Exemplo: CAMINHO_TFRECORD_SAFRINHA = '/content/drive/MyDrive/Tese_IA_Jussara/seu_arquivo.tfrecord.gz'
CAMINHO_TFRECORD_SAFRINHA = None

# Busca também em subpastas e de forma independente de maiúsculas/minúsculas.
BUSCAR_EM_SUBPASTAS = True

# Se nenhum nome tiver "safrinha"/"segunda safra", usa o melhor TFRecord disponível
# (preferindo 2025 e MASSIVE) em vez de interromper o notebook imediatamente.
PERMITIR_FALLBACK_GENERICO = True

TERMOS_SAFRINHA = ('safrinha', 'segunda_safra', 'segunda-safra')

def caminho_normalizado(arquivo):
    return os.path.basename(arquivo).lower().replace(' ', '_')

def parece_tfrecord(arquivo):
    nome = caminho_normalizado(arquivo)
    return 'tfrecord' in nome or nome.endswith(('.record', '.record.gz', '.records', '.records.gz'))

def calcular_prioridade(arquivo):
    nome = caminho_normalizado(arquivo)
    prioridade = 0
    if any(termo in nome for termo in TERMOS_SAFRINHA) or ('segunda' in nome and 'safra' in nome):
        prioridade += 100
    if '2025' in nome:
        prioridade += 50
    if 'massive' in nome:
        prioridade += 20
    return prioridade

def buscar_tfrecords():
    if not os.path.isdir(pasta_base):
        raise FileNotFoundError(f'pasta_base não existe ou não está acessível: {pasta_base}')

    arquivos = []
    if BUSCAR_EM_SUBPASTAS:
        for raiz, _, nomes in os.walk(pasta_base):
            for nome in nomes:
                arquivo = os.path.join(raiz, nome)
                if parece_tfrecord(arquivo):
                    arquivos.append(arquivo)
    else:
        for arquivo in glob.glob(os.path.join(pasta_base, '*')):
            if os.path.isfile(arquivo) and parece_tfrecord(arquivo):
                arquivos.append(arquivo)

    arquivos = list(dict.fromkeys(arquivos))
    arquivos.sort(key=lambda arquivo: (calcular_prioridade(arquivo), os.path.getmtime(arquivo)), reverse=True)
    return arquivos

def mostrar_candidatos(arquivos, titulo):
    if not arquivos:
        return
    print(titulo)
    for i, arquivo in enumerate(arquivos[:10], start=1):
        print(f'  {i}. prioridade={calcular_prioridade(arquivo):03d} | {arquivo}')

def localizar_tfrecord_safrinha():
    if CAMINHO_TFRECORD_SAFRINHA:
        if os.path.exists(CAMINHO_TFRECORD_SAFRINHA):
            print('✅ Usando TFRecord informado manualmente em CAMINHO_TFRECORD_SAFRINHA.')
            return CAMINHO_TFRECORD_SAFRINHA
        raise FileNotFoundError(f'O caminho informado manualmente não existe: {CAMINHO_TFRECORD_SAFRINHA}')

    candidatos = buscar_tfrecords()
    candidatos_safrinha = [arquivo for arquivo in candidatos if calcular_prioridade(arquivo) >= 100]
    if candidatos_safrinha:
        mostrar_candidatos(candidatos_safrinha, '✅ Candidatos da safrinha encontrados:')
        return candidatos_safrinha[0]

    if PERMITIR_FALLBACK_GENERICO and candidatos:
        print('⚠️ Nenhum arquivo com SAFRINHA/segunda safra no nome foi encontrado.')
        print('⚠️ Vou usar o TFRecord mais bem ranqueado como fallback, preferindo 2025 e MASSIVE.')
        print('⚠️ Se este não for o arquivo correto, preencha CAMINHO_TFRECORD_SAFRINHA manualmente.')
        mostrar_candidatos(candidatos, '📋 TFRecords encontrados:')
        return candidatos[0]

    raise FileNotFoundError(
        'Nenhum TFRecord foi encontrado em pasta_base ou subpastas. '
        'Confira se o Google Drive foi montado, se pasta_base está correta e se o arquivo contém "tfrecord" ou "record" no nome. '
        'Você também pode preencher CAMINHO_TFRECORD_SAFRINHA com o caminho completo.'
    )

caminho_arquivo = localizar_tfrecord_safrinha()
print(f"📂 Lendo dados da {NOME_CENARIO}: {caminho_arquivo}")

# Parâmetros
KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 40
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

# Nomes isolados para não sobrescrever os modelos/checkpoints anteriores.
checkpoint_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Jussara_{NOME_CENARIO}.keras')
final_path = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{NOME_CENARIO}_FINAL.keras')

# --- PIPELINE DE DADOS ---
def parse_and_process(example_proto):
    features_dict = {
        band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
    }
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Contagem rápida para não dar erro de tamanho
print('🔢 Verificando tamanho do arquivo...')
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f'✅ Total de amostras: {N_REAL}')

if N_REAL < 2:
    raise ValueError('O TFRecord precisa ter pelo menos 2 amostras para separar treino e validação.')

N_TRAIN = max(1, int(N_REAL * 0.8))
N_VAL = N_REAL - N_TRAIN
if N_VAL == 0:
    N_TRAIN -= 1
    N_VAL = 1

full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(parse_and_process)

train_ds = full_dataset.take(N_TRAIN).cache().shuffle(N_TRAIN).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
print(f'🧪 Treino: {N_TRAIN} amostras | Validação: {N_VAL} amostras')

# --- MODELO U-NET ---
def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    p1 = layers.MaxPooling2D()(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    p2 = layers.MaxPooling2D()(c2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)

    # Decoder
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)

    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)

    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])

model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(INPUT_BANDS)))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 🛡️ SALVAMENTO AUTOMÁTICO (SEGURANÇA) ---
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=False,
    verbose=1
)

print(f'🔥 Iniciando retreinamento da {NOME_CENARIO}...')
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb]
)

# Salvamento final definitivo
model.save(final_path)
print(f'✅ SUCESSO! Modelo final da {NOME_CENARIO} salvo em: {final_path}')


## 🔎 Prova visual da safrinha 2025

Execute a célula abaixo depois do treinamento para carregar o arquivo salvo e visualizar exemplos com a predição da IA.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from google.colab import drive

# 1. Montar Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print('--- INICIANDO PROVA REAL DA SAFRINHA 2025 ---')

# 2. Localizar o modelo salvo da safrinha 2025
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
NOME_CENARIO = 'Safrinha_2025'
caminho_modelo = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{NOME_CENARIO}_FINAL.keras')

if os.path.exists(caminho_modelo):
    print(f'✅ Arquivo do modelo encontrado: {caminho_modelo}')
    model = tf.keras.models.load_model(caminho_modelo)
    print('✅ Modelo carregado na memória com sucesso!')
else:
    raise FileNotFoundError(f'O arquivo .keras não foi encontrado: {caminho_modelo}')

# 3. Carregar dados da safrinha 2025 para testar
# Use o mesmo caminho manual da célula de treinamento, se necessário.
# Se a busca automática não encontrar o arquivo correto, cole o caminho completo aqui.
# Exemplo: CAMINHO_TFRECORD_SAFRINHA = '/content/drive/MyDrive/Tese_IA_Jussara/seu_arquivo.tfrecord.gz'
CAMINHO_TFRECORD_SAFRINHA = None

# Busca também em subpastas e de forma independente de maiúsculas/minúsculas.
BUSCAR_EM_SUBPASTAS = True

# Se nenhum nome tiver "safrinha"/"segunda safra", usa o melhor TFRecord disponível
# (preferindo 2025 e MASSIVE) em vez de interromper o notebook imediatamente.
PERMITIR_FALLBACK_GENERICO = True

TERMOS_SAFRINHA = ('safrinha', 'segunda_safra', 'segunda-safra')

def caminho_normalizado(arquivo):
    return os.path.basename(arquivo).lower().replace(' ', '_')

def parece_tfrecord(arquivo):
    nome = caminho_normalizado(arquivo)
    return 'tfrecord' in nome or nome.endswith(('.record', '.record.gz', '.records', '.records.gz'))

def calcular_prioridade(arquivo):
    nome = caminho_normalizado(arquivo)
    prioridade = 0
    if any(termo in nome for termo in TERMOS_SAFRINHA) or ('segunda' in nome and 'safra' in nome):
        prioridade += 100
    if '2025' in nome:
        prioridade += 50
    if 'massive' in nome:
        prioridade += 20
    return prioridade

def buscar_tfrecords():
    if not os.path.isdir(pasta_base):
        raise FileNotFoundError(f'pasta_base não existe ou não está acessível: {pasta_base}')

    arquivos = []
    if BUSCAR_EM_SUBPASTAS:
        for raiz, _, nomes in os.walk(pasta_base):
            for nome in nomes:
                arquivo = os.path.join(raiz, nome)
                if parece_tfrecord(arquivo):
                    arquivos.append(arquivo)
    else:
        for arquivo in glob.glob(os.path.join(pasta_base, '*')):
            if os.path.isfile(arquivo) and parece_tfrecord(arquivo):
                arquivos.append(arquivo)

    arquivos = list(dict.fromkeys(arquivos))
    arquivos.sort(key=lambda arquivo: (calcular_prioridade(arquivo), os.path.getmtime(arquivo)), reverse=True)
    return arquivos

def mostrar_candidatos(arquivos, titulo):
    if not arquivos:
        return
    print(titulo)
    for i, arquivo in enumerate(arquivos[:10], start=1):
        print(f'  {i}. prioridade={calcular_prioridade(arquivo):03d} | {arquivo}')

def localizar_tfrecord_safrinha():
    if CAMINHO_TFRECORD_SAFRINHA:
        if os.path.exists(CAMINHO_TFRECORD_SAFRINHA):
            print('✅ Usando TFRecord informado manualmente em CAMINHO_TFRECORD_SAFRINHA.')
            return CAMINHO_TFRECORD_SAFRINHA
        raise FileNotFoundError(f'O caminho informado manualmente não existe: {CAMINHO_TFRECORD_SAFRINHA}')

    candidatos = buscar_tfrecords()
    candidatos_safrinha = [arquivo for arquivo in candidatos if calcular_prioridade(arquivo) >= 100]
    if candidatos_safrinha:
        mostrar_candidatos(candidatos_safrinha, '✅ Candidatos da safrinha encontrados:')
        return candidatos_safrinha[0]

    if PERMITIR_FALLBACK_GENERICO and candidatos:
        print('⚠️ Nenhum arquivo com SAFRINHA/segunda safra no nome foi encontrado.')
        print('⚠️ Vou usar o TFRecord mais bem ranqueado como fallback, preferindo 2025 e MASSIVE.')
        print('⚠️ Se este não for o arquivo correto, preencha CAMINHO_TFRECORD_SAFRINHA manualmente.')
        mostrar_candidatos(candidatos, '📋 TFRecords encontrados:')
        return candidatos[0]

    raise FileNotFoundError(
        'Nenhum TFRecord foi encontrado em pasta_base ou subpastas. '
        'Confira se o Google Drive foi montado, se pasta_base está correta e se o arquivo contém "tfrecord" ou "record" no nome. '
        'Você também pode preencher CAMINHO_TFRECORD_SAFRINHA com o caminho completo.'
    )

caminho_dados = localizar_tfrecord_safrinha()
print(f'📂 Validando com dados de: {caminho_dados}')

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'

def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Pega apenas 1 lote de até 10 imagens
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP')
dataset = dataset.map(parse_fast).batch(10).take(1)

# 4. Gerar previsões
print('🔮 Gerando previsões com o modelo carregado...')
imgs, labels = next(iter(dataset))
preds = model.predict(imgs)

# 5. Visualizar
n_exemplos = min(5, imgs.shape[0])
plt.figure(figsize=(15, 3 * n_exemplos))
print('\nLEGENDA: Esquerda=Satélite | Meio=Gabarito | Direita=O que a IA Aprendeu')

for i in range(n_exemplos):
    # Satélite (NDVI do primeiro momento)
    plt.subplot(n_exemplos, 3, i * 3 + 1)
    plt.imshow(imgs[i][:, :, 2], cmap='RdYlGn', vmin=0, vmax=0.8)
    plt.axis('off')
    if i == 0:
        plt.title('Satélite (NDVI)')

    # Gabarito
    plt.subplot(n_exemplos, 3, i * 3 + 2)
    plt.imshow(labels[i][:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0:
        plt.title('Gabarito Real')

    # Predição da IA
    plt.subplot(n_exemplos, 3, i * 3 + 3)
    plt.imshow(preds[i][:, :, 0], cmap='magma', vmin=0, vmax=1)
    plt.axis('off')
    if i == 0:
        plt.title('IA Safrinha 2025')

plt.tight_layout()
plt.show()
